In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def read_paymentdetail_data():
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    from pyspark.sql.functions import current_timestamp
  
    paymentdetail_schema = StructType([
    StructField("id",StringType()),
    StructField("srno",StringType()),
    StructField("paymentdate",StringType()),
    StructField("amount",StringType()),
    StructField("cheque",StringType()),
    StructField("refno",StringType()),
    StructField("payingparty",StringType()),
    StructField("receivingparty",StringType()),
    StructField("bankname",StringType()),
    StructField("details",StringType())
    ])

    rawpayment_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{tracking_br_path}/schema')
        .option('header','true')
        .schema(paymentdetail_schema)
        .load(tracking_raw_path+'/paymentdetail/')
        .withColumn("ingestion_time",current_timestamp())
    )
    return rawpayment_stream

In [0]:
from pyspark.sql.functions import col, substring

def write_paymentdetail_Data(StreamingDF):
    StreamingDF = StreamingDF.withColumn(
        "id",
        substring(col("id"), 1, 100)
    )
    write_Stream = (
        StreamingDF.writeStream
        .format("delta")
        .option("checkpointLocation", f"{tracking_br_path}/checkpoint/paymentdetail/")
        .trigger(availableNow=True)
        .toTable("dev.tracking_br.paymentdetail")
    )
    write_Stream.awaitTermination()

In [0]:
df_paymemt = read_paymentdetail_data()
write_paymentdetail_Data(df_paymemt)
